# CS 5326 – PA2: SFT + DPO on Qwen2.5-0.5B (Kaggle T4 ×2 orchestrator)

This notebook only **orchestrates**: every long-running step is a script in `scripts/`, and every stage writes
durable artifacts under `/kaggle/working/model-alignment/outputs/` as it goes.

**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All (Commit)**.
A full run takes roughly 7–8 h (inside Kaggle's 12 h limit). Set `SMOKE_TEST = True` first for a ~15–25 min end-to-end check.

| Stage | Manual task | GPU | Output |
|---|---|---|---|
| Setup, versions, tests | Task 1 | – | test log |
| Data prep (UltraChat 30k/1k, HH 15k/1k, UltraFeedback 15k/1k) | §6 | – | `outputs/data/` |
| Baseline generations + persona LoRA-DPO (background) | Tasks 2, 5 | GPU 1 | `generations/`, `persona_lora/` |
| SFT | Task 3 | GPU 0 | `sft/{best,final}` |
| Sanity check (3 prompts) | Task 6 | GPU 0 | printed |
| HH-RLHF DPO, then UltraFeedback DPO | Task 4 | GPU 0 policy / GPU 1 reference | `hh_dpo/`, `ultrafeedback_dpo/` |
| SFT/DPO generations, blinded pair files | Task 6 | both | `generations/`, `eval/*_pairs.jsonl` |
| Judging (only if this kernel has Kaggle model-proxy access; else use `kaggle_judge.ipynb`) | Task 7 | – | `eval/*_judged*.json*` |
| Plots, results tables, download bundle | Task 8 / §16 | – | `plots/`, `report/`, `submission_bundle.zip` |

Rerunning is safe: finished stages are skipped (`DONE` markers), and generation and judging resume per item.

In [ ]:
# ============================== configuration ==============================
REPO_URL = "https://github.com/alihasnaink/model-alignment.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/model-alignment"

SMOKE_TEST = False          # True = tiny end-to-end run (~15-25 min) to validate the pipeline before the real run
OVERLAP_GPU_WORK = True     # run generation / persona LoRA on GPU 1 while GPU 0 trains (saves ~1 h)
RESUME_FROM = None          # e.g. "/kaggle/input/<previous-version-output>/model-alignment/outputs" to skip finished work
ZIP_BEST_CHECKPOINTS = False  # extra ~6 GB zip of the SFT/HH/UF best checkpoints (Kaggle /kaggle/working limit is ~20 GB)

SEED = 42
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
PERSONA_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
JUDGE_MODEL = "google/gemini-3.6-flash"

# Guide configuration (manual §9.2 / §10.4). Every value is recorded in each run's run_config.json.
SFT_HP = dict(max_length=512, batch_size=2, gradient_accumulation=16, learning_rate=2e-5, warmup_ratio=0.03,
              weight_decay=0.1, max_grad_norm=1.0, epochs=1, eval_every=100)
DPO_HP = dict(max_length=512, batch_size=1, gradient_accumulation=32, learning_rate=1e-6, beta=0.1,
              optimizer="adamw", weight_decay=0.0, max_grad_norm=1.0, epochs=1, eval_every=100)
PERSONA_HP = dict(epochs=3, learning_rate=1e-4, batch_size=2, gradient_accumulation=4, beta=0.1,
                  lora_rank=16, lora_alpha=32, lora_dropout=0.05)
# Same decoding for every model in a comparison: greedy + modest repetition penalty (small models loop otherwise).
GEN_HP = dict(batch_size=16, max_new_tokens=512, repetition_penalty=1.1, no_repeat_ngram_size=0)

## Task 1 – Repository and Kaggle setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

if not Path(REPO_DIR, "submission_adapter.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Using existing repository: {REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
repo = Path(REPO_DIR).resolve()
os.chdir(repo)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

# Keep Kaggle's CUDA-matched PyTorch: requirements.txt does not list torch, and torchao is removed because
# older Kaggle builds conflict with PEFT.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import datasets, peft, torch, transformers, trl

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("trl / peft  :", trl.__version__, "/", peft.__version__)
print("GPUs        :", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Enable a GPU accelerator."
if not SMOKE_TEST:
    assert torch.cuda.device_count() >= 2, "Select the GPU T4 x2 accelerator (policy on cuda:0, reference on cuda:1)."

# Public adapter smoke tests + our edge-case and end-to-end script tests. Stop here if anything fails.
test_env = {**os.environ, "PYTHONPATH": str(repo), "CUDA_VISIBLE_DEVICES": "0"}  # 1 GPU: no DataParallel in the TRL test
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], check=True, env=test_env)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], check=True, env=test_env)

In [ ]:
# ============================== helpers ==============================
import json, shutil, threading, time, zipfile
from IPython.display import FileLink, Image, Markdown, display

OUT = repo / ("outputs_smoke" if SMOKE_TEST else "outputs")
DATA, GEN, EVAL, LOGS = OUT / "data", OUT / "generations", OUT / "eval", OUT / "logs"
for d in (DATA, GEN, EVAL, LOGS):
    d.mkdir(parents=True, exist_ok=True)
ALPACA_EVAL = repo / "eval_sets/alpaca_eval_300_seed42.json"
HH_EVAL = repo / "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_EVAL = repo / "eval_sets/persona_style_eval.json"

if RESUME_FROM and Path(RESUME_FROM).exists():
    # Copy only files we do not already have, so newer local work is never overwritten.
    for src in Path(RESUME_FROM).rglob("*"):
        dest = OUT / src.relative_to(RESUME_FROM)
        if src.is_file() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
    print("Resumed artifacts from", RESUME_FROM)


def flags(hp):
    return [x for key, value in hp.items() for x in (f"--{key}", value)]


def env_for(gpus):
    env = {**os.environ, "PYTHONPATH": str(repo), "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    if gpus is not None:
        env["CUDA_VISIBLE_DEVICES"] = gpus
    return env


def run(cmd, gpus=None, log_name=None):
    # Run a script in the foreground, streaming output to the cell and to outputs/logs/<log_name>.log.
    cmd = [str(c) for c in cmd]
    log_path = LOGS / f"{log_name or Path(cmd[1]).stem}.log"
    print(f"$ [{'GPU ' + gpus if gpus else 'all GPUs'}] {' '.join(cmd)}", flush=True)
    start = time.time()
    with log_path.open("a") as log, subprocess.Popen(cmd, cwd=repo, env=env_for(gpus), stdout=subprocess.PIPE,
                                                       stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}); see {log_path}")
    print(f"   finished in {(time.time() - start) / 60:.1f} min", flush=True)


class Background:
    # Runs a list of commands sequentially on one GPU in a thread; output goes to a log file only.
    def __init__(self, name, commands, gpus):
        self.name, self.commands, self.gpus, self.error = name, commands, gpus, None
        self.log = LOGS / f"background_{name}.log"
        if OVERLAP_GPU_WORK:
            self.thread = threading.Thread(target=self._work, daemon=True)
            self.thread.start()
            print(f"Started background track '{name}' on GPU {gpus} ({len(commands)} commands) -> {self.log}")
        else:
            self.thread = None
            for cmd in commands:
                run(cmd, gpus=None, log_name=f"background_{name}")

    def _work(self):
        try:
            with self.log.open("a") as log:
                for cmd in self.commands:
                    cmd = [str(c) for c in cmd]
                    log.write(f"\n$ {' '.join(cmd)}\n"); log.flush()
                    start = time.time()
                    code = subprocess.run(cmd, cwd=repo, env=env_for(self.gpus), stdout=log, stderr=subprocess.STDOUT).returncode
                    log.write(f"   exit {code} after {(time.time() - start) / 60:.1f} min\n"); log.flush()
                    if code != 0:
                        raise RuntimeError(f"{cmd[1]} exited with {code}")
        except Exception as error:
            self.error = error

    def wait(self):
        if self.thread is None:
            return
        print(f"Waiting for background track '{self.name}'...", flush=True)
        self.thread.join()
        print("".join(self.log.read_text().splitlines(keepends=True)[-25:]))
        if self.error:
            raise RuntimeError(f"background track '{self.name}' failed: {self.error}. Full log: {self.log}")
        print(f"Background track '{self.name}' finished OK.")


def done(stage_dir):
    return (Path(stage_dir) / "DONE").exists()


def gen_cmd(model, inputs, output, generator, dataset, extra=()):
    return ["python", "scripts/generate.py", "--model", model, "--inputs", inputs, "--output", output,
            "--generator", generator, "--dataset", dataset, *flags(GEN_HP), *SMOKE_GEN, *extra]


# Tiny sizes for SMOKE_TEST; empty for the real run.
SMOKE_PREP = ["--train_size", 200, "--validation_size", 50] if SMOKE_TEST else []
SMOKE_SFT = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_DPO = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_GEN = ["--limit", 6, "--max_new_tokens", 64] if SMOKE_TEST else []
SMOKE_PERSONA = ["--max_steps", 4] if SMOKE_TEST else []
print("Outputs ->", OUT)

## Data preparation (manual §6)
* **SFT**: UltraChat-200k `train_sft`, first user→assistant exchange of each conversation, seeded 30,000 / 1,000 split.
* **HH-RLHF DPO**: `Anthropic/hh-rlhf` single-turn pairs, 15,000 train / 1,000 validation (disjoint, seed 42).
* **UltraFeedback DPO**: `ultrafeedback_binarized` `train_prefs` → 15,000 train, `test_prefs` → 1,000 validation.
* Both preference sets: non-empty, chosen ≠ rejected, deduplicated, formatted prompt ≤ 256 tokens (the manual's max prompt length).
* Fixed eval sets live in `eval_sets/` (committed): AlpacaEval 300 / seed 42, the provided HH eval set, and the persona style set.

In [ ]:
ULTRACHAT_DIR, HH_DIR, UF_DIR = DATA / "ultrachat", DATA / "hh", DATA / "ultrafeedback"
if not (ULTRACHAT_DIR / "sft_valid.jsonl").exists():
    run(["python", "scripts/prepare_ultrachat.py", "--output_dir", ULTRACHAT_DIR, "--seed", SEED, *SMOKE_PREP])
if not (HH_DIR / "hh_valid.jsonl").exists():
    run(["python", "scripts/prepare_preferences.py", "--track", "hh", "--output_dir", HH_DIR, "--seed", SEED, *SMOKE_PREP])
if not (UF_DIR / "ultrafeedback_valid.jsonl").exists():
    run(["python", "scripts/prepare_preferences.py", "--track", "ultrafeedback", "--output_dir", UF_DIR, "--seed", SEED, *SMOKE_PREP])
if not ALPACA_EVAL.exists():
    run(["python", "scripts/prepare_alpaca_eval.py", "--output", ALPACA_EVAL, "--num_prompts", 300, "--seed", SEED])

# Keep a copy of the exact eval subsets next to the results (manual: save the exact subsets used).
shutil.copytree(repo / "eval_sets", DATA / "eval_sets", dirs_exist_ok=True)
for path in sorted(DATA.rglob("*.json*")):
    if path.suffix == ".jsonl" or path.parent.name == "eval_sets":
        rows = sum(1 for _ in path.open()) if path.suffix == ".jsonl" else len(json.loads(path.read_text()))
        print(f"{rows:>7}  {path.relative_to(OUT)}")

## Tasks 2 + 5 (background on GPU 1) and Task 3 – SFT (GPU 0)
While SFT trains on GPU 0, GPU 1 runs: base-model generations on the AlpacaEval subset (Task 2) and the HH eval set,
then the TRL `DPOTrainer` + LoRA persona run (Task 5) and its base/LoRA generations on the persona style set.

In [ ]:
PERSONA_OUT = OUT / "persona_lora"
track_a = [
    gen_cmd(BASE_MODEL, ALPACA_EVAL, GEN / "base_alpaca_eval.json", "qwen2.5-0.5b-base", "alpaca_eval"),
    gen_cmd(BASE_MODEL, HH_EVAL, GEN / "base_hh_eval.json", "qwen2.5-0.5b-base", "hh_rlhf_style_eval"),
    gen_cmd(PERSONA_MODEL, PERSONA_EVAL, GEN / "persona_base_persona_eval.json", "smollm2-360m-instruct",
            "persona_style_eval", ["--prompt_format", "chat"]),
]
if not done(PERSONA_OUT):
    track_a.append(["python", "scripts/train_persona_lora.py", "--model", PERSONA_MODEL, "--output_dir", PERSONA_OUT,
                    "--seed", SEED, *flags(PERSONA_HP), *SMOKE_PERSONA])
track_a.append(gen_cmd(PERSONA_MODEL, PERSONA_EVAL, GEN / "persona_lora_persona_eval.json", "smollm2-360m-instruct-lora-dpo",
                       "persona_style_eval", ["--prompt_format", "chat", "--adapter", PERSONA_OUT / "final"]))
bg_a = Background("baseline_and_persona", track_a, gpus="1" if torch.cuda.device_count() > 1 else "0")

In [ ]:
SFT_OUT = OUT / "sft"
if done(SFT_OUT):
    print("SFT already done:", json.loads((SFT_OUT / "summary.json").read_text()))
else:
    run(["python", "scripts/train_sft.py", "--model", BASE_MODEL,
         "--train", ULTRACHAT_DIR / "sft_train.jsonl", "--validation", ULTRACHAT_DIR / "sft_valid.jsonl",
         "--output_dir", SFT_OUT, "--seed", SEED, *flags(SFT_HP), *SMOKE_SFT], gpus="0")
print(json.loads((SFT_OUT / "summary.json").read_text()))

In [ ]:
bg_a.wait()
print(json.dumps(json.loads((PERSONA_OUT / "summary.json").read_text()), indent=2))

## Task 6 sanity check – a few SFT generations before full-subset generation
Check by eye that the checkpoint path, tokenizer, prompt format, EOS behaviour, and generation length look right.

In [ ]:
sanity = GEN / "sanity" / "sft_best_sanity.json"
run(gen_cmd(SFT_OUT / "best", ALPACA_EVAL, sanity, "sft-sanity", "alpaca_eval",
            ["--limit", 3, "--print_examples", 3, "--no-resume"]), gpus="0")
rows = json.loads(sanity.read_text())
assert all(r["output"].strip() for r in rows), "SFT produced empty outputs - check prompt format / EOS handling"
print("token counts:", [r["num_new_tokens"] for r in rows], "(512 = hit the cap; mostly lower means EOS works)")

## Task 4 – two from-scratch DPO runs from the SFT `best` checkpoint
Policy (fp32 + fp16 autocast) on `cuda:0`, frozen fp16 reference on `cuda:1`. β = 0.1, LR 1e-6 constant, effective batch 32.
Best checkpoint = highest validation reward accuracy. While each run trains, GPU 1 also generates the outputs that are already possible.

In [ ]:
def run_dpo(name, data_dir, track):
    out = OUT / name
    if done(out):
        print(f"{name} already done:", json.loads((out / "summary.json").read_text()))
        return
    run(["python", "scripts/train_dpo.py", "--sft_checkpoint", SFT_OUT / "best",
         "--train", data_dir / f"{track}_train.jsonl", "--validation", data_dir / f"{track}_valid.jsonl",
         "--output_dir", out, "--run_name", name, "--seed", SEED, *flags(DPO_HP), *SMOKE_DPO])
    print(json.loads((out / "summary.json").read_text()))

HH_OUT, UF_OUT = OUT / "hh_dpo", OUT / "ultrafeedback_dpo"
bg_b = Background("sft_generations", [
    gen_cmd(SFT_OUT / "best", ALPACA_EVAL, GEN / "sft_alpaca_eval.json", "qwen2.5-0.5b-sft", "alpaca_eval"),
    gen_cmd(SFT_OUT / "best", HH_EVAL, GEN / "sft_hh_eval.json", "qwen2.5-0.5b-sft", "hh_rlhf_style_eval"),
], gpus="1" if torch.cuda.device_count() > 1 else "0")
run_dpo("hh_dpo", HH_DIR, "hh")
bg_b.wait()

In [ ]:
bg_c = Background("hh_dpo_generations", [
    gen_cmd(HH_OUT / "best", HH_EVAL, GEN / "hh_dpo_hh_eval.json", "qwen2.5-0.5b-hh-dpo", "hh_rlhf_style_eval"),
], gpus="1" if torch.cuda.device_count() > 1 else "0")
run_dpo("ultrafeedback_dpo", UF_DIR, "ultrafeedback")
bg_c.wait()
run(gen_cmd(UF_OUT / "best", ALPACA_EVAL, GEN / "ultrafeedback_dpo_alpaca_eval.json", "qwen2.5-0.5b-ultrafeedback-dpo",
            "alpaca_eval"), gpus="0")

## Task 6 – blinded pairwise comparison files
A/B order is randomized per pair (seed 42); the model mapping is stored as hidden metadata and never shown to the judge.

In [ ]:
PAIRS = [
    ("hh_dpo_vs_sft", GEN / "sft_hh_eval.json", "sft", GEN / "hh_dpo_hh_eval.json", "hh_dpo"),
    ("ultrafeedback_dpo_vs_sft", GEN / "sft_alpaca_eval.json", "sft", GEN / "ultrafeedback_dpo_alpaca_eval.json", "ultrafeedback_dpo"),
    ("persona_lora_vs_base", GEN / "persona_base_persona_eval.json", "base_instruct", GEN / "persona_lora_persona_eval.json", "lora_dpo"),
    ("sft_vs_base_alpaca", GEN / "base_alpaca_eval.json", "base", GEN / "sft_alpaca_eval.json", "sft"),  # extra analysis
]
for stem, baseline, baseline_name, candidate, candidate_name in PAIRS:
    run(["python", "scripts/build_pairwise.py", "--baseline", baseline, "--baseline_name", baseline_name,
         "--candidate", candidate, "--candidate_name", candidate_name, "--output", EVAL / f"{stem}_pairs.jsonl", "--seed", SEED])

## Task 7 – judging
Kaggle's free judge access (`kaggle_benchmarks`) only works in a notebook set up as a **utility benchmark**, which normally
is not this GPU notebook. If this kernel has no model-proxy credentials, judging is skipped here: save this version, then run
**`kaggle_judge.ipynb`** with this notebook's output attached (see README). In `SMOKE_TEST` mode an offline mock judge
exercises the code path instead.

In [ ]:
kaggle_judge_ready = "MODEL_PROXY_URL" in os.environ and "MODEL_PROXY_API_KEY" in os.environ
if kaggle_judge_ready:
    run(["python", "scripts/run_judging.py", "--eval_dir", EVAL, "--judge_model", JUDGE_MODEL, "--backend", "kaggle",
         "--batch_size", 5, "--sleep_seconds", 5, "--max_retries", 3])
elif SMOKE_TEST:
    run(["python", "scripts/run_judging.py", "--eval_dir", EVAL, "--backend", "mock", "--judge_model", "mock-longer-wins"])
else:
    print("No Kaggle model-proxy credentials in this kernel -> judging deferred to kaggle_judge.ipynb.")

## Task 8 – plots, results tables, and the download bundle

In [ ]:
run(["python", "scripts/plot_metrics.py", "--outputs", OUT])
run(["python", "scripts/summarize_results.py", "--outputs", OUT])
for png in sorted((OUT / "plots").glob("*.png")):
    display(Markdown(f"**{png.name}**")); display(Image(filename=str(png)))

In [ ]:
CHECKPOINT_DIRS = {"best", "final", "trainer"}

def zip_outputs(dest, include_checkpoints=False):
    # Zip OUT, leaving out full model checkpoints (kept in the saved notebook output instead).
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(OUT.rglob("*")):
            rel = path.relative_to(OUT)
            in_ckpt = any(part in CHECKPOINT_DIRS for part in rel.parts)
            is_large_data = rel.parts[0] == "data" and path.suffix == ".jsonl"
            keep_adapter = rel.parts[0] == "persona_lora" and "final" in rel.parts  # LoRA adapter is small
            if path.is_file() and (not in_ckpt or keep_adapter or include_checkpoints) and not is_large_data:
                zf.write(path, Path(OUT.name) / rel)
    print(f"{dest}  {dest.stat().st_size / 1e6:.1f} MB")

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else repo.parent
bundle = WORK / ("submission_bundle_smoke.zip" if SMOKE_TEST else "submission_bundle.zip")
zip_outputs(bundle)
if ZIP_BEST_CHECKPOINTS:
    with zipfile.ZipFile(WORK / "best_checkpoints.zip", "w", zipfile.ZIP_STORED) as zf:
        for run_dir in (SFT_OUT, HH_OUT, UF_OUT):
            for path in (run_dir / "best").rglob("*"):
                zf.write(path, Path(run_dir.name) / "best" / path.relative_to(run_dir / "best"))

# Download links (also available from the notebook's Output tab after Save Version).
os.chdir(WORK)
display(Markdown("### Downloads"))
for name in (bundle.name, "best_checkpoints.zip"):
    if (WORK / name).exists():
        display(FileLink(name))
os.chdir(repo)
subprocess.run(["du", "-sh", str(OUT), str(WORK)], check=False)

In [ ]:
summary = {name: json.loads((OUT / name / "summary.json").read_text()) for name in ("sft", "hh_dpo", "ultrafeedback_dpo", "persona_lora")
           if (OUT / name / "summary.json").exists()}
print(json.dumps(summary, indent=2, default=str))
judged = sorted(EVAL.glob("*_judged_summary.json"))
print("Judge summaries:", [p.name for p in judged] or "none yet - run kaggle_judge.ipynb")
display(Markdown((OUT / "report" / "results.md").read_text()[:6000]))
print("\nAll stages finished. Save Version now to preserve /kaggle/working (checkpoints + outputs).")